# Praktikum 5

Asyam Sakhi Maulana
NIM 240535600363

In [8]:
!pip install --quiet spacy nltk pandas
!python -m spacy download en_core_web_sm

import nltk
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

import spacy
import pandas as pd
from spacy.training.iob_utils import offsets_to_biluo_tags, biluo_to_iob

nlp = spacy.load("en_core_web_sm")

corpus = [
    (47, "The poor schnooks at Procter & Gamble and Nike and Colgate-Palmolive are spending $4 billion a year on advertising."),
    (49, "The little text-only ads on Google perform well while the banners on Yahoo do poorly."),
    (100, "Archie McPhee did it in retail with their product selection, and Starbucks determined how to redefine what a cup of coffee meant, in a way very different from JetBlue."),
    (101, "Tiffany's blue box is a slogan without words, just like the Hooters name and logo or the funky hipness of Apple's industrial design."),
    (101, "The Leaning Tower of Pisa sees millions of visitors every year, but it is much harder to tell them about the Pantheon in Rome."),
]

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
tokens_per_sentence = []
for halaman, text in corpus:
    toks_case = nltk.word_tokenize(text)          # case-sensitive
    toks_lower = nltk.word_tokenize(text.lower())  # pembanding
    tokens_per_sentence.append(toks_case)
    print(f"[hlm {halaman}] {text}")
    print(f"  Token (case-sensitive): {toks_case}")
    print(f"  Token (lowercase)     : {toks_lower}")
    print()

print("""
Analisis: kapitalisasi adalah sinyal penting untuk POS Tagging dan NER karena
huruf kapital di TENGAH kalimat (bukan di awal kalimat) hampir selalu menandai
proper noun (nama diri) — mis. "Procter", "Google", "Starbucks", "Rome". Kalau
teks di-lowercase semua, penanda visual ini hilang: "google" jadi tidak bisa
dibedakan dari kata benda biasa hanya dari bentuknya, sehingga POS tagger lebih
sulit memberi tag NNP/NNPS dan model NER pretrained (yang dilatih pada teks
case-sensitive) kehilangan salah satu fitur paling kuat untuk mendeteksi
entitas. Ini juga alasan Soal 1 meminta tokenisasi case-sensitive dipertahankan
untuk tahap POS Tagging (Soal 2) dan NER (Soal 3), bukan versi lowercase.
""")

[hlm 47] The poor schnooks at Procter & Gamble and Nike and Colgate-Palmolive are spending $4 billion a year on advertising.
  Token (case-sensitive): ['The', 'poor', 'schnooks', 'at', 'Procter', '&', 'Gamble', 'and', 'Nike', 'and', 'Colgate-Palmolive', 'are', 'spending', '$', '4', 'billion', 'a', 'year', 'on', 'advertising', '.']
  Token (lowercase)     : ['the', 'poor', 'schnooks', 'at', 'procter', '&', 'gamble', 'and', 'nike', 'and', 'colgate-palmolive', 'are', 'spending', '$', '4', 'billion', 'a', 'year', 'on', 'advertising', '.']

[hlm 49] The little text-only ads on Google perform well while the banners on Yahoo do poorly.
  Token (case-sensitive): ['The', 'little', 'text-only', 'ads', 'on', 'Google', 'perform', 'well', 'while', 'the', 'banners', 'on', 'Yahoo', 'do', 'poorly', '.']
  Token (lowercase)     : ['the', 'little', 'text-only', 'ads', 'on', 'google', 'perform', 'well', 'while', 'the', 'banners', 'on', 'yahoo', 'do', 'poorly', '.']

[hlm 100] Archie McPhee did it in reta

In [10]:
for (halaman, text), toks in zip(corpus, tokens_per_sentence):
    tagged = nltk.pos_tag(toks)
    df = pd.DataFrame(tagged, columns=["Token", "POS"])
    kandidat = [t for t, tag in tagged if tag in ("NNP", "NNPS")]
    print(f"[hlm {halaman}] {text}")
    print(df.to_string(index=False))
    print("Kandidat entitas (NNP/NNPS):", kandidat)
    print()

[hlm 47] The poor schnooks at Procter & Gamble and Nike and Colgate-Palmolive are spending $4 billion a year on advertising.
            Token POS
              The  DT
             poor  JJ
         schnooks NNS
               at  IN
          Procter NNP
                &  CC
           Gamble NNP
              and  CC
             Nike NNP
              and  CC
Colgate-Palmolive NNP
              are VBP
         spending VBG
                $   $
                4  CD
          billion  CD
                a  DT
             year  NN
               on  IN
      advertising  NN
                .   .
Kandidat entitas (NNP/NNPS): ['Procter', 'Gamble', 'Nike', 'Colgate-Palmolive']

[hlm 49] The little text-only ads on Google perform well while the banners on Yahoo do poorly.
    Token POS
      The  DT
   little  JJ
text-only  JJ
      ads NNS
       on  IN
   Google NNP
  perform  NN
     well  RB
    while  IN
      the  DT
  banners NNS
       on  IN
    Yahoo NNP
       do VBP
   po

In [11]:
docs = [nlp(text) for _, text in corpus]

for (halaman, text), doc in zip(corpus, docs):
    print(f"[hlm {halaman}] {text}")
    if doc.ents:
        for ent in doc.ents:
            print(f"   {ent.text!r:35} -> {ent.label_}")
    else:
        print("   (tidak ada entitas terdeteksi)")
    print()

[hlm 47] The poor schnooks at Procter & Gamble and Nike and Colgate-Palmolive are spending $4 billion a year on advertising.
   'Procter & Gamble'                  -> ORG
   'Nike'                              -> ORG
   'Colgate-Palmolive'                 -> ORG
   '$4 billion'                        -> MONEY

[hlm 49] The little text-only ads on Google perform well while the banners on Yahoo do poorly.
   'Google'                            -> ORG
   'Yahoo'                             -> ORG

[hlm 100] Archie McPhee did it in retail with their product selection, and Starbucks determined how to redefine what a cup of coffee meant, in a way very different from JetBlue.
   'Archie McPhee'                     -> PERSON
   'Starbucks'                         -> ORG
   'JetBlue'                           -> ORG

[hlm 101] Tiffany's blue box is a slogan without words, just like the Hooters name and logo or the funky hipness of Apple's industrial design.
   'Tiffany'                         

In [12]:
def to_bio(doc):
    return [f"{t.ent_iob_}-{t.ent_type_}" if t.ent_iob_ != "O" else "O" for t in doc]

for (halaman, text), doc in zip(corpus, docs):
    bio_pred = to_bio(doc)
    df = pd.DataFrame({"Token": [t.text for t in doc], "BIO (prediksi)": bio_pred})
    print(f"[hlm {halaman}] {text}")
    print(df.to_string(index=False))
    print()

[hlm 47] The poor schnooks at Procter & Gamble and Nike and Colgate-Palmolive are spending $4 billion a year on advertising.
      Token BIO (prediksi)
        The              O
       poor              O
   schnooks              O
         at              O
    Procter          B-ORG
          &          I-ORG
     Gamble          I-ORG
        and              O
       Nike          B-ORG
        and              O
    Colgate          B-ORG
          -          I-ORG
  Palmolive          I-ORG
        are              O
   spending              O
          $        B-MONEY
          4        I-MONEY
    billion        I-MONEY
          a              O
       year              O
         on              O
advertising              O
          .              O

[hlm 49] The little text-only ads on Google perform well while the banners on Yahoo do poorly.
  Token BIO (prediksi)
    The              O
 little              O
   text              O
      -              O
   only         

In [13]:
gold_spec = [
    [("Procter & Gamble", "ORG"), ("Nike", "ORG"), ("Colgate-Palmolive", "ORG"), ("$4 billion", "MONEY")],
    [("Google", "ORG"), ("Yahoo", "ORG")],
    [("Archie McPhee", "ORG"), ("Starbucks", "ORG"), ("JetBlue", "ORG")],
    [("Tiffany", "ORG"), ("Hooters", "ORG"), ("Apple", "ORG")],
    [("The Leaning Tower of Pisa", "FAC"), ("Pantheon", "FAC"), ("Rome", "GPE")],
]

def make_offsets(text, spec):
    offsets = []
    for phrase, label in spec:
        start = text.find(phrase)
        if start == -1:
            raise ValueError(f"'{phrase}' tidak ditemukan di: {text}")
        offsets.append((start, start + len(phrase), label))
    return offsets

all_gold_iob, all_pred_iob = [], []
total_tp = total_fp = total_fn = 0

for (halaman, text), doc, spec in zip(corpus, docs, gold_spec):
    offsets = make_offsets(text, spec)
    gold_iob = biluo_to_iob(offsets_to_biluo_tags(doc, offsets))
    pred_iob = to_bio(doc)
    status = ["Benar" if g == p else "Salah" for g, p in zip(gold_iob, pred_iob)]

    df = pd.DataFrame({
        "Token": [t.text for t in doc],
        "Gold": gold_iob,
        "Prediksi": pred_iob,
        "Status": status,
    })
    print(f"[hlm {halaman}] {text}")
    print(df.to_string(index=False))

    gold_ents = set(spec)
    pred_ents = set((ent.text, ent.label_) for ent in doc.ents)
    tp, fp, fn = gold_ents & pred_ents, pred_ents - gold_ents, gold_ents - pred_ents
    print(f"  TP ({len(tp)}): {tp}")
    print(f"  FP ({len(fp)}): {fp}")
    print(f"  FN ({len(fn)}): {fn}\n")

    all_gold_iob += gold_iob
    all_pred_iob += pred_iob
    total_tp += len(tp); total_fp += len(fp); total_fn += len(fn)

[hlm 47] The poor schnooks at Procter & Gamble and Nike and Colgate-Palmolive are spending $4 billion a year on advertising.
      Token    Gold Prediksi Status
        The       O        O  Benar
       poor       O        O  Benar
   schnooks       O        O  Benar
         at       O        O  Benar
    Procter   B-ORG    B-ORG  Benar
          &   I-ORG    I-ORG  Benar
     Gamble   I-ORG    I-ORG  Benar
        and       O        O  Benar
       Nike   B-ORG    B-ORG  Benar
        and       O        O  Benar
    Colgate   B-ORG    B-ORG  Benar
          -   I-ORG    I-ORG  Benar
  Palmolive   I-ORG    I-ORG  Benar
        are       O        O  Benar
   spending       O        O  Benar
          $ B-MONEY  B-MONEY  Benar
          4 I-MONEY  I-MONEY  Benar
    billion I-MONEY  I-MONEY  Benar
          a       O        O  Benar
       year       O        O  Benar
         on       O        O  Benar
advertising       O        O  Benar
          .       O        O  Benar
  TP (4): {

In [14]:
tok_correct = sum(g == p for g, p in zip(all_gold_iob, all_pred_iob))
tok_total = len(all_gold_iob)

accuracy = tok_correct / tok_total
precision = total_tp / (total_tp + total_fp)
recall = total_tp / (total_tp + total_fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"Accuracy (token-level)   = {tok_correct}/{tok_total} = {accuracy:.4f}")
print(f"Precision (entity-level) = {total_tp}/({total_tp}+{total_fp}) = {precision:.4f}")
print(f"Recall (entity-level)    = {total_tp}/({total_tp}+{total_fn}) = {recall:.4f}")
print(f"F1-Score                 = {f1:.4f}")

print(f"""
Interpretasi: Accuracy token-level cukup tinggi ({accuracy:.2%}) karena mayoritas
token dalam kelima kalimat memang berlabel "O" (bukan bagian entitas), dan model
hampir selalu benar menebak "O" untuk token-token tersebut. Tapi Precision
({precision:.2%}) dan Recall ({recall:.2%}) jauh lebih rendah, karena keduanya hanya
menghitung token/entitas yang MEMANG bagian dari entitas — dan di sinilah model
sering salah: "Archie McPhee" dan "Hooters" salah dikira PERSON (bukan ORG), dan
pada kalimat "Leaning Tower of Pisa" model malah menambahkan entitas ekstra yang
tidak relevan (WORK_OF_ART, CARDINAL, DATE) serta salah label FAC menjadi
WORK_OF_ART/ORG. Kalau evaluasi hanya pakai accuracy, kesalahan-kesalahan pada
entitas ini akan tenggelam oleh banyaknya token "O" yang benar — persis seperti
peringatan di materi kuliah: accuracy saja BELUM CUKUP untuk mengevaluasi NER,
karena ia tidak sensitif terhadap kegagalan mengenali entitas yang justru jadi
inti tujuan NER.
""")

Accuracy (token-level)   = 114/126 = 0.9048
Precision (entity-level) = 11/(11+6) = 0.6471
Recall (entity-level)    = 11/(11+4) = 0.7333
F1-Score                 = 0.6875

Interpretasi: Accuracy token-level cukup tinggi (90.48%) karena mayoritas
token dalam kelima kalimat memang berlabel "O" (bukan bagian entitas), dan model
hampir selalu benar menebak "O" untuk token-token tersebut. Tapi Precision
(64.71%) dan Recall (73.33%) jauh lebih rendah, karena keduanya hanya
menghitung token/entitas yang MEMANG bagian dari entitas — dan di sinilah model
sering salah: "Archie McPhee" dan "Hooters" salah dikira PERSON (bukan ORG), dan
pada kalimat "Leaning Tower of Pisa" model malah menambahkan entitas ekstra yang
tidak relevan (WORK_OF_ART, CARDINAL, DATE) serta salah label FAC menjadi
WORK_OF_ART/ORG. Kalau evaluasi hanya pakai accuracy, kesalahan-kesalahan pada
entitas ini akan tenggelam oleh banyaknya token "O" yang benar — persis seperti
peringatan di materi kuliah: accuracy saja BELUM CUKUP